# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [4]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, alternate arching your back up (cat) and letting it sag down (cow) for 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds. Repeat 8-12 times.\n\nThese exercises are gentle stretching and strengthening movements that can help alleviate lower back discomfort and prevent future episodes.'

In [11]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in maintaining overall health. It is essential for physical recovery, as during sleep the body repairs tissues and regenerates. Sleep also supports mental well-being and cognitive functions like memory consolidation and learning. Adequate sleep, typically 7-9 hours per night, helps boost the immune system, regulate hormones related to growth and appetite, and reduce the risk of health issues such as headaches, stress, and chronic conditions. Practicing good sleep hygiene—like maintaining a consistent sleep schedule, creating a comfortable sleep environment, and establishing relaxing bedtime routines—can improve sleep quality and, consequently, overall health.'

In [12]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

"Some natural remedies for stress include practicing deep breathing (inhaling for 4 counts, holding for 4, exhaling for 4), doing progressive muscle relaxation by tensing and releasing muscle groups, using grounding techniques (naming things you see, hear, feel, smell, and taste), taking short walks in nature, and listening to calming music.\n\nFor headaches, natural remedies mentioned are staying well-hydrated by drinking water, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, gently massaging the temples and neck, and using essential oils like peppermint or lavender. Additionally, maintaining a regular sleep schedule can help prevent headaches.\n\nIf you need further assistance or personalized advice, it's best to consult a healthcare professional."

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees. Alternate between arching your back up (-cat) and letting it sag down (cow). Perform 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold each extension for about 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent. Flatten your back against the floor by tightening your abs and tilting your pelvis upward. Hold for 10 seconds, and repeat 8-12 times.\n\nThese exercises gently stretch and strengthen the muscles supporting your lower back, which can help alleviate pain and prevent future issues.'

In [16]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a crucial role in overall health. Maintaining a consistent sleep schedule, creating a comfortable sleep environment, and practicing good sleep hygiene—such as avoiding screens before bed, limiting caffeine later in the day, and establishing relaxing routines—are important for achieving quality sleep. Adequate sleep supports immune function, mental health, and physical recovery, contributing to your overall well-being. Conversely, poor sleep or insomnia can negatively impact health, making it essential to prioritize healthy sleep habits.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include relaxation techniques such as deep breathing and progressive muscle relaxation, meditation, and herbal teas like chamomile or valerian root. Additionally, staying well-hydrated, ensuring good sleep hygiene, and managing stress through activities like mindfulness can help reduce headaches and stress.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:
**Example Query:** `Please provide instructions for the cat-cow stretch.`

This query would be better suited for BM25 than embeddings because it includes specific keywords that appear in the wellness document, which will be ranked higher with BM25. Embeddings focuses on semantic similarity, and this query could result in other stretches listed in the wellness doc achieving high ranks, even though we only wanted information on the specific cat-cow stretch.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [18]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [19]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Based on the information provided, some exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch**: Start on your hands and knees. Alternate between arching your back up (like a cat) and letting it sag down (like a cow). Aim for 10-15 repetitions.\n- **Bird Dog**: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold each extension for about 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Pelvic Tilts**: Lie on your back with knees bent. Flatten your back against the floor by tightening your abdominal muscles and tilting your pelvis slightly upward. Hold for 10 seconds and repeat 8-12 times.\n\nThese exercises are gentle and can help alleviate discomfort and prevent future episodes of lower back pain. However, it's always best to consult with a healthcare professional before starting any new exercise routine, especially if you have existing health conditions."

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is essential for physical repair, mental well-being, and cognitive functions. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep—typically 7-9 hours per night—is vital for maintaining good health, supporting immune function, and ensuring proper functioning of the brain and body. Poor sleep or sleep disorders like insomnia can negatively affect health, leading to issues such as impaired memory, weakened immunity, and increased risk of chronic conditions.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include drinking water to stay hydrated, applying cold or warm compresses to the head or neck, resting in a dark and quiet room, gentle massage of the temples and neck, using peppermint or lavender essential oils, maintaining a regular sleep schedule, practicing deep breathing or progressive muscle relaxation, engaging in grounding techniques, taking a short walk in nature, and listening to calming music.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [23]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [24]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [25]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help alleviate lower back pain include:\n\n1. Cat-Cow Stretch: Start on your hands and knees, then alternate between arching your back up (cat position) and letting it sag down (cow position). Perform 10-15 repetitions.\n\n2. Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold each extension for 5 seconds, then switch sides. Complete 10 repetitions per side.\n\n3. Partial Crunches: Lie on your back with knees bent, arms crossed over your chest, tighten your stomach muscles, and raise your shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n4. Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat on the floor. Hold for 15-30 seconds, then switch legs.\n\n5. Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThe

In [26]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical repair, mental well-being, and cognitive functioning. During sleep, your body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate and quality sleep (generally 7-9 hours for adults) is essential for maintaining a strong immune system, managing stress, and ensuring proper bodily functions. Poor sleep or sleep disturbances, such as insomnia, can negatively impact health, leading to issues like fatigue, stress, weakened immunity, and increased risk of chronic conditions. Therefore, maintaining good sleep hygiene and creating an optimal sleep environment are important steps toward overall health and wellness.'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water to stay hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of temples and neck\n- Using peppermint or lavender essential oils\n- Maintaining a regular sleep schedule\n- Practicing deep breathing exercises\n- Engaging in progressive muscle relaxation\n- Taking short walks, preferably in nature\n- Listening to calming music\n\nThese strategies can help alleviate symptoms naturally and promote overall well-being.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:
Because a user's query is often not optimized for retrieval, generating many different versions of the user's query can pull additional context from documents that may not have been ranked as relevant to the original query and improve recall due to more relevant documents retrieved (those from the original query and those from the different versions). For example, a generated query of "What are the benefits of 8 hours of sleep?" could help to retrieve additional relevant documents for our example "How does sleep affect overall health?" query in this notebook.

However, if the different versions of the user's original query aren't relevant to the original query and irrelevant context is included, this can also add noise when generating the response.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [28]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [29]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [30]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [31]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [32]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [33]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'To help with lower back pain, the following exercises are recommended:\n\n1. **Cat-Cow Stretch:** Start on hands and knees, alternate arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n2. **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n3. **Partial Crunches:** Lie on your back with knees bent, cross arms over your chest, and raise shoulders off the floor by tightening your stomach muscles. Do 8-12 repetitions.\n4. **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n5. **Pelvic Tilts:** Lie on your back with knees bent, tighten your abs and tilt your pelvis upward slightly to flatten your back against the floor. Hold for 10 seconds, repeat 8-12 times.\n\nThese exercises can help alleviate discomfort and prevent future episodes when done gen

In [34]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is crucial for physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adults typically need 7-9 hours of sleep per night, and good sleep quality is supported by maintaining healthy sleep habits such as a consistent sleep schedule, creating a relaxing bedtime routine, and creating an optimal sleep environment. Poor sleep can lead to issues like fatigue, low energy, headaches, and dizziness, while sufficient, quality sleep promotes better physical health, mental clarity, and emotional stability.'

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing exercises, engaging in progressive muscle relaxation, mindfulness or meditation, taking short walks in nature, listening to calming music, and using essential oils such as peppermint or lavender. These techniques can help reduce tension, promote relaxation, and alleviate headache symptoms.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [36]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [37]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [38]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\nThese exercises are gentle and aimed at stretching and strengthening the muscles to help alleviate lower bac

In [39]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical, mental, and cognitive functions. During sleep, your body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep—typically 7-9 hours per night—helps boost immune function, improve mood, enhance learning and memory, and reduce the risk of various health issues. Maintaining good sleep hygiene and an optimal sleep environment can promote better sleep quality, which in turn contributes positively to your overall well-being.'

In [40]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises (e.g., box breathing)\n- Progressive muscle relaxation\n- Grounding techniques (naming objects around you)\n- Taking short walks, preferably in nature\n- Listening to calming music\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of temples and neck\n- Using essential oils such as peppermint or lavender\n- Maintaining a regular sleep schedule\n\nAdditionally, managing stress through practices like mindfulness, meditation, and ensuring good sleep hygiene can help reduce the frequency and intensity of headaches caused by stress.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [41]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [42]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [43]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [44]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [45]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [46]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides.\n\nThese gentle stretching and strengthening exercises can help alleviate discomfort and improve back health.'

In [47]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical, mental, and cognitive functions. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep (typically 7-9 hours for adults) helps maintain immune function, supports mental well-being, reduces the risk of chronic diseases, and promotes physical recovery. Poor sleep quality or insufficient sleep can lead to fatigue, weakened immune response, impaired cognitive function, increased stress levels, and higher susceptibility to health problems. Therefore, prioritizing good sleep hygiene and ensuring restful sleep are essential for overall health and wellness.'

In [48]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises (e.g., inhaling for 4 counts, holding, exhaling, and holding again)\n- Progressive muscle relaxation (tensing and releasing muscle groups)\n- Grounding techniques (naming objects you see, hear, feel, smell, and taste)\n- Taking short walks, especially in nature\n- Listening to calming music\n- Drinking water to stay hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Using essential oils such as peppermint or lavender\n- Practicing mindfulness meditation or deep breathing exercises\n\nThese methods can help alleviate headaches and reduce stress naturally.'

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:
In this case, semantic chunking might result in larger chunks due to similar content / smaller distances between consecutive sentences, which could end up including irrelevant information in these larger chunks. We may look to adjust this by changing the thresholding method (such as gradient to detect shifts in topic or lowering standard deviation) or by lowering the percentile threshold to create more breakpoints.

---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [84]:
### YOUR CODE HERE

# Enable LangSmith tracing for detailed cost and latency tracking across all chain invocations
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "advanced-retrieval-evaluation"

In [85]:
# Ragas testset generation requires LangchainLLMWrapper / LangchainEmbeddingsWrapper
# to bridge LangChain models with the Ragas testset API
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/var/folders/zx/dvl4qkvd4jg2rfjvsrfyhcj40000gn/T/ipykernel_19139/4102110639.py:5: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
/var/folders/zx/dvl4qkvd4jg2rfjvsrfyhcj40000gn/T/ipykernel_19139/4102110639.py:6: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


In [86]:
# Build the knowledge graph for testset generation and save it so the
# expensive transforms (headline extraction, NER, embeddings) only run once.
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.transforms import default_transforms, apply_transforms
from langchain_text_splitters import RecursiveCharacterTextSplitter

KG_PATH = "wellness_kg.json"

if os.path.exists(KG_PATH):
    kg = KnowledgeGraph.load(KG_PATH)
    print(f"Loaded existing knowledge graph ({len(kg.nodes)} nodes)")
else:
    # default_transforms requires >500 tokens per document for the headline/split path
    # that enables multi-hop relationships; 3000-char chunks (~613 tokens each) satisfy this
    kg_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=200)
    kg_docs = kg_splitter.split_documents(raw_docs)

    kg = KnowledgeGraph()
    for doc in kg_docs:
        kg.nodes.append(
            Node(
                type=NodeType.DOCUMENT,
                properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
            )
        )
    transforms = default_transforms(documents=kg_docs, llm=generator_llm, embedding_model=generator_embeddings)
    apply_transforms(kg, transforms)
    kg.save(KG_PATH)
    print(f"Built and saved knowledge graph ({len(kg.nodes)} nodes) to {KG_PATH}")

Loaded existing knowledge graph (15 nodes)


In [87]:
import pandas as pd
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
)

TESTSET_PATH = "wellness_testset.csv"

if os.path.exists(TESTSET_PATH):
    testset_df = pd.read_csv(TESTSET_PATH)
    print(f"Loaded existing testset ({len(testset_df)} examples)")
else:
    generator = TestsetGenerator(
        llm=generator_llm,
        embedding_model=generator_embeddings,
        knowledge_graph=kg,
    )
    # 50% single-hop (fact from one chunk), 25% multi-hop abstract (reasoning across chunks),
    # 25% multi-hop specific (fact requiring multiple chunks)
    query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm),  0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm),  0.25),
    ]
    testset = generator.generate(testset_size=15, query_distribution=query_distribution)
    testset_df = testset.to_pandas()
    testset_df.to_csv(TESTSET_PATH, index=False)
    print(f"Generated and saved testset ({len(testset_df)} examples) to {TESTSET_PATH}")


In [88]:
testset_df.head()

In [89]:
import time
import pandas as pd
from langsmith import Client
from langsmith.evaluation import evaluate as ls_evaluate
from ragas import EvaluationDataset, evaluate as ragas_evaluate, RunConfig
from ragas.metrics import (
    LLMContextRecall,
    Faithfulness,
    FactualCorrectness,
    ResponseRelevancy,
    ContextEntityRecall,
    NoiseSensitivity,
)

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
custom_run_config = RunConfig(timeout=360)

# Retriever-specific: LLMContextRecall, ContextEntityRecall, NoiseSensitivity
# End-to-end:         Faithfulness, FactualCorrectness, ResponseRelevancy
ragas_metrics = [
    LLMContextRecall(),
    Faithfulness(),
    FactualCorrectness(),
    ResponseRelevancy(),
    ContextEntityRecall(),
    NoiseSensitivity(),
]

# Upload testset to LangSmith; recreate each run to stay in sync with the current testset
ls_client = Client()
DATASET_NAME = "wellness-guide-golden-dataset"

if ls_client.has_dataset(dataset_name=DATASET_NAME):
    ls_client.delete_dataset(dataset_name=DATASET_NAME)

ls_client.create_dataset(
    dataset_name=DATASET_NAME,
    description="15 wellness queries (single-hop + multi-hop) generated by Ragas",
)
ls_client.create_examples(
    dataset_name=DATASET_NAME,
    examples=[
        {"inputs": {"question": row["user_input"]}, "outputs": {"reference": row["reference"]}}
        for _, row in testset_df.iterrows()
    ],
)
print(f"Created LangSmith dataset '{DATASET_NAME}' with {len(testset_df)} examples")

/var/folders/zx/dvl4qkvd4jg2rfjvsrfyhcj40000gn/T/ipykernel_19139/2688592059.py:6: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
/var/folders/zx/dvl4qkvd4jg2rfjvsrfyhcj40000gn/T/ipykernel_19139/2688592059.py:6: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/var/folders/zx/dvl4qkvd4jg2rfjvsrfyhcj40000gn/T/ipykernel_19139/2688592059.py:6: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import (

Created LangSmith dataset 'wellness-guide-golden-dataset' with 16 examples


In [91]:
retrieval_configs = {
    "naive":                  naive_retrieval_chain,
    "bm25":                   bm25_retrieval_chain,
    "contextual_compression": contextual_compression_retrieval_chain,
    "multi_query":            multi_query_retrieval_chain,
    "parent_document":        parent_document_retrieval_chain,
    "ensemble":               ensemble_retrieval_chain,
}

# contextual_compression and ensemble both call CohereRerank (Trial key: 10 req/min)
cohere_chains = {"contextual_compression", "ensemble"}

# Ragas result columns that are not metric scores
non_metric_cols = {"user_input", "retrieved_contexts", "response", "reference"}

def evaluate_retriever(name):
    chain = retrieval_configs[name]
    print(f"Evaluating: {name}...")

    def target(inputs):
        question = inputs["question"]
        result = chain.invoke({"question": question})
        responses_cache[question] = {
            "answer": result["response"].content,
            "contexts": [doc.page_content for doc in result["context"]],
        }
        if name in cohere_chains:
            time.sleep(7)  # Cohere Trial key: 10 req/min
        return responses_cache[question]

    # Phase 1: run chain inside ls_evaluate — LangSmith traces latency + token cost
    responses_cache = {}
    experiment_results = ls_evaluate(
        target,
        data=DATASET_NAME,
        evaluators=[],
        experiment_prefix=name,
        max_concurrency=1 if name in cohere_chains else 4,
    )

    # Phase 2: build Ragas EvaluationDataset from cached responses
    eval_rows = [
        {
            "user_input": question,
            "retrieved_contexts": responses_cache[question]["contexts"],
            "response": responses_cache[question]["answer"],
            "reference": row["reference"],
        }
        for _, row in testset_df.iterrows()
        for question in [row["user_input"]]
        if question in responses_cache
    ]
    eval_dataset = EvaluationDataset.from_pandas(pd.DataFrame(eval_rows))

    # Phase 3: score all 6 metrics concurrently with ragas.evaluate()
    ragas_result = ragas_evaluate(
        dataset=eval_dataset,
        metrics=ragas_metrics,
        llm=evaluator_llm,
        run_config=custom_run_config,
    )
    ragas_df = ragas_result.to_pandas()
    actual_metric_names = [c for c in ragas_df.columns if c not in non_metric_cols]
    scores_by_question = {row["user_input"]: row for _, row in ragas_df.iterrows()}

    # Phase 4: attach Ragas scores to Phase 1 LangSmith runs via feedback
    for run_result in experiment_results:
        run_id = run_result["run"].id
        question = run_result["example"].inputs["question"]
        if question in scores_by_question:
            scores_row = scores_by_question[question]
            for metric_name in actual_metric_names:
                score = scores_row[metric_name]
                if pd.notna(score):
                    ls_client.create_feedback(
                        run_id=run_id,
                        key=metric_name,
                        score=float(score),
                    )

    results[name] = experiment_results
    print(f"  Done: {name}")

results = {}


In [92]:
evaluate_retriever("naive")

In [93]:
evaluate_retriever("bm25")

In [94]:
evaluate_retriever("contextual_compression")

In [96]:
evaluate_retriever("multi_query")

In [97]:
evaluate_retriever("parent_document")

In [98]:
evaluate_retriever("ensemble")

In [114]:
# ── 1. Collect all feedback in one bulk call ────────────────────────────────
rows = []
all_run_ids = [
    str(run_result['run'].id)
    for experiment in results.values()
    for run_result in experiment
]
all_feedback = list(ls_client.list_feedback(run_ids=all_run_ids))
feedback_by_run = {}
for fb in all_feedback:
    feedback_by_run.setdefault(str(fb.run_id), []).append(fb)

for retriever_name, experiment in results.items():
    for run_result in experiment:
        run = run_result['run']
        run_id = str(run.id)
        latency = (
            (run.end_time - run.start_time).total_seconds()
            if run.end_time and run.start_time else None
        )
        row = {'retriever': retriever_name, 'latency_s': latency}
        for fb in feedback_by_run.get(run_id, []):
            row[fb.key] = fb.score
        rows.append(row)

df_runs = pd.DataFrame(rows)

# ── 2. Average per retriever ─────────────────────────────────────────────────
avg = df_runs.groupby('retriever').mean(numeric_only=True)

def clean_col(col):
    col = col.split('(')[0].replace('_', ' ').title()
    return col.replace('Llm', 'LLM')

order = ['naive', 'bm25', 'contextual_compression', 'multi_query', 'parent_document', 'ensemble']
order = [r for r in order if r in avg.index]

display_labels = {
    'naive':                  'Naive',
    'bm25':                   'BM25',
    'contextual_compression': 'Contextual Compression',
    'multi_query':            'Multi-Query',
    'parent_document':        'Parent Document',
    'ensemble':               'Ensemble',
}

ragas_cols = [c for c in avg.columns if c != 'latency_s']
summary = avg.loc[order, ragas_cols].rename(columns={c: clean_col(c) for c in ragas_cols})
summary['Latency (s)'] = avg.loc[order, 'latency_s'].round(2).values
# Cost pulled from LangSmith UI (not available via SDK run objects)
summary['Cost (USD)'] = [{'naive': 0.0034, 'bm25': 0.0017, 'contextual_compression': 0.0019,
                           'multi_query': 0.0047, 'parent_document': 0.0027, 'ensemble': 0.0066
                          }.get(r, float('nan')) for r in order]
summary.index = [display_labels[r] for r in order]
summary.index.name = 'Retriever'
summary = summary.round(4)

# ── 3. Color scale (lower is better for: Noise Sensitivity, Latency, Cost) ──
def col_color(series, reverse=False):
    vmin, vmax = series.min(), series.max()
    def css(v):
        try:
            v = float(v)
        except (TypeError, ValueError):
            return ''
        if pd.isna(v):
            return ''
        norm = max(0.0, min(1.0, (v - vmin) / (vmax - vmin) if vmax != vmin else 0.5))
        if reverse:
            norm = 1.0 - norm
        r = int(220 * (1.0 - norm)) if norm >= 0.5 else 220
        g = 180 if norm >= 0.5 else int(220 * norm * 2)
        return f'background-color: rgb({r},{g},80); color: black'
    return [css(v) for v in series]

inverted_cols = {'Noise Sensitivity', 'Latency (s)', 'Cost (USD)'}
styled = summary.style
for col in summary.columns:
    styled = styled.apply(col_color, subset=[col], reverse=(col in inverted_cols))

ragas_display_cols = [c for c in summary.columns if c not in inverted_cols]
display(
    styled
    .format('{:.4f}', subset=ragas_display_cols)
    .format('{:.2f}s', subset=['Latency (s)'])
    .format('${:.6f}', subset=['Cost (USD)'])
    .set_caption('Average scores per retriever across the golden dataset')
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '13px'), ('font-weight', 'bold'), ('padding-bottom', '8px')]},
        {'selector': 'th',      'props': [('font-size', '11px'), ('text-align', 'center'), ('padding', '4px 8px')]},
        {'selector': 'td',      'props': [('font-size', '11px'), ('text-align', 'center'), ('padding', '5px 10px')]},
    ])
)


,Noise Sensitivity,Context Entity Recall,Answer Relevancy,Factual Correctness,Faithfulness,Context Recall,Latency (s),Cost (USD)
Retriever,,,,,,,,
Naive,0.128700,0.1652,0.9442,0.7844,0.9722,1.0000,2.03s,$0.003400
BM25,0.160000,0.0675,0.7700,0.6050,0.9171,0.8125,1.23s,$0.001700
Contextual Compression,0.113000,0.1423,0.8918,0.7594,0.9090,0.9792,8.78s,$0.001900
Multi-Query,0.109900,0.1802,0.9547,0.8188,0.8959,1.0000,3.27s,$0.004700
Parent Document,0.225300,0.2375,0.9514,0.8025,0.8966,1.0000,1.77s,$0.002700
Ensemble,0.275700,0.1397,0.9508,0.8394,0.9345,1.0000,10.82s,$0.006600


## Analysis

Ensemble retrieval is the best overall retriever for our wellness use case, considering it had the highest factual correctness (0.84), second best faithfulness (0.93), and perfect context recall. Combining all five retrievers helps to capture the most complete and accurate information. The latency (10.82s) and cost ($0.0066) are real drawbacks, but the performance justifies it due to our health domain where both answer accuracy and groundedness matter most. The higher noise sensitivity (0.28) is worth noting, but the strong faithfulness score mitigates the risk of irrelevant context leading to hallucinated health advice.

Naive retrieval was a surprisingly strong baseline: it had the best faithfulness by a wide margin (0.97), perfect context recall, and reasonable factual correctness (0.78) at low latency (2.03s) and cost ($0.0034). The high faithfulness is especially valuable in a health context because it means the system sticks closely to the source material rather than generating plausible-sounding but unsupported wellness advice. Multi-query retrieval achieved the best answer relevancy (0.95), lowest noise sensitivity (0.11), and second-best factual correctness (0.82), but it had the lowest faithfulness of all retrievers (0.90) - this is likely due to the generated query strategy pulling in more context which improves coverage, but may lead the LLM to extrapolate beyond what was actually retrieved. For wellness content, that tradeoff is hard to justify over ensemble or naive.

Parent document retrieval had the best context entity recall (0.24) by a clear margin and fast latency (1.77s), making it the best value option, though its noise sensitivity (0.23) is notable. Contextual Compression had high latency (8.78s) from Cohere reranking without meaningful improvement over naive on this dataset. BM25 underperformed across all metrics: worst factual correctness (0.61), worst context recall (0.81), worst answer relevancy (0.77), confirming that keyword retrieval alone is not the right fit for nuanced health and wellness queries.